# The learned curve

What the source network converged to, how it got there, and what the harmonic
count K does. CPU only; no datasets and no checkpoints are needed -- everything
below comes from the five-number artifact and from
`results/tables/coeff_traj_seed1.json`.


In [1]:
import json
import matplotlib.pyplot as plt
import numpy as np
import torch

from left.activations import FAct, gelu_init_coefficients
from left.activations.fourier_fit import fourier_series_eval

ARTIFACT = "../artifacts/fact/imagenet1k_vitd6_seed1.json"
art = json.load(open(ARTIFACT))
art["a0"], art["a"], art["b"]

(0.21817652881145477,
 [0.11933743208646774, -0.42374375462532043],
 [0.8315256237983704, -0.5918471813201904])

## Learned vs GELU vs the initialisation

The curve starts as the Fourier fit of GELU (dashed) and is learned from there.
The learned shape is non-monotone and periodic -- structurally unlike every
conventional activation it is compared against.

In [2]:
t = np.linspace(-6, 6, 1201)
init = gelu_init_coefficients(K=2)

learned = fourier_series_eval(t, art["a0"], art["a"], art["b"], art["w"])
started = fourier_series_eval(t, init["a0"], init["a"], init["b"], init["w"])
gelu = torch.nn.functional.gelu(torch.from_numpy(t)).numpy()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(t, gelu, color="0.6", lw=2, label="GELU")
ax.plot(t, started, "--", color="tab:orange", label="FAct init (GELU Fourier fit, K=2)")
ax.plot(t, learned, color="tab:blue", lw=2, label="FAct learned on ImageNet-1K")
ax.axhline(0, color="0.9", lw=0.8, zorder=0)
ax.axvline(0, color="0.9", lw=0.8, zorder=0)
ax.set_xlabel("pre-activation $t$"); ax.set_ylabel(r"$\varphi(t)$")
ax.set_title("One curve, shared by every block of the source network")
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()

## How it got there

The per-epoch coefficient trajectory of the source run (seed 1, 100 epochs).
The coefficients move early and then settle -- the curve is converged, not
still drifting when training stopped.

In [3]:
traj = json.load(open("../results/tables/coeff_traj_seed1.json"))["seed1"]
traj = np.asarray(traj)
labels = ["$a_0$", "$a_1$", "$a_2$", "$b_1$", "$b_2$"]

fig, ax = plt.subplots(figsize=(7, 4))
for i, lab in enumerate(labels):
    ax.plot(np.arange(1, len(traj) + 1), traj[:, i], label=lab)
ax.set_xlabel("epoch"); ax.set_ylabel("coefficient")
ax.set_title("Source-run coefficient trajectory (ImageNet-1K, seed 1)")
ax.legend(frameon=False, ncol=5, fontsize=9)
fig.tight_layout()

print("final:", traj[-1].round(4), "\nartifact:",
      np.array([art["a0"], *art["a"], *art["b"]])[[0, 1, 2, 3, 4]].round(4))

final: [ 0.2182  0.1193 -0.4237  0.8315 -0.5918] 
artifact: [ 0.2182  0.1193 -0.4237  0.8315 -0.5918]


## Harmonic count

K is a knob, not a hard-coded 2. More harmonics fit GELU more closely at
initialisation. The paper reports K = 2 only; other values are supported but
not evaluated.

In [4]:
tt = torch.linspace(-3, 3, 601)
gelu_t = torch.nn.functional.gelu(tt)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(tt, gelu_t, color="0.6", lw=2.5, label="GELU")
for K in (2, 5, 9, 21):
    act = FAct(K=K)
    err = (act(tt) - gelu_t).abs().mean().item()
    ax.plot(tt, act(tt).detach(), lw=1.2, label=f"K={K}  (MAE {err:.3f})")
ax.set_xlabel("$t$"); ax.set_ylabel(r"$\varphi(t)$")
ax.set_title("GELU-fit initialisation at increasing K")
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()

## The transfer step, end to end

Five numbers out of a file and into a network that shares nothing else with the
source.

In [5]:
from left.models.mlp_mixer import build_mlpmixer
from left.transfer import activation_parameter_count, assert_frozen, find_activations

act = FAct.from_artifact(ARTIFACT)
model = build_mlpmixer("cifar10", act, embed_dim=76)

assert_frozen(model)
print(act)
print("distinct activation instances:", len(find_activations(model)))
print("trainable activation parameters:", activation_parameter_count(model))
print("total model parameters:", sum(p.numel() for p in model.parameters()))

FAct(K=2, w=1.000000, frozen=True, ref=None, a0=0.218177, a=[0.119337, -0.423744], b=[0.831526, -0.591847])
distinct activation instances: 1
trainable activation parameters: 0
total model parameters: 108367
